# Games. Minimax. Alpha-Beta Pruning. and Tic-Tac-Toe

## 1. Games

### 1.1 Types of Games

- **Cooperative vs Non-cooperative** – in cooperative games, players can form binding
agreements and coalitions, so the analysis focuses on what groups can achieve and how
they split the payoff. In non-cooperative games, each player acts alone and agreements
cannot be enforced.
- **Symmetric vs Asymmetric** – in a symmetric game, payoffs depend only on the strategies
played, not on who plays them. In an asymmetric game, players have different strategy
sets or payoffs.
- **Zero-sum vs Non-zero-sum** – in a zero-sum game, one player's gain is exactly the
other's loss, so the total payoff stays constant (e.g. chess, poker). In a non-zero-sum
game, the total can change, so all players may gain or lose together.
- **Simultaneous vs Sequential** – in simultaneous games, players choose at the same time
or without knowing the others' choices. These games are usually written in normal
(matrix) form. In sequential games, players take turns and know at least something about
earlier moves. These games are written in extensive (tree) form.
- **Perfect vs Imperfect information** – with perfect information, every player knows all
moves made so far (e.g. chess, Go). With imperfect information, some earlier moves or
cards are hidden (e.g. poker).
- **Combinatorial games** – two-player sequential games with perfect information and no
chance, where the difficulty comes from the huge number of possible moves
(e.g. chess, Go, Nim).
- **Infinitely long games** – games that can continue forever, so the winner is decided by
the whole infinite sequence of moves rather than by reaching a terminal state. They are
mostly studied in mathematics and logic.
- **Games with chance** – games with random events, modeled as moves made by "Nature"
(e.g. dice rolls in backgammon, shuffled cards). Players maximize their expected payoff.
- **Bayesian games** – games with incomplete information, where players are unsure of the
others' payoffs or "types" (a player's private information). Each player holds a belief
(a probability distribution) over the possible types and chooses the strategy that
maximizes their expected payoff under that belief (e.g. auctions, poker).

### 1.2 Perfect and Complete Information

**Complete information** means knowing the game itself: who the players are, what actions
each can take, and what each player's payoffs are. If any player is unsure of someone
else's payoffs or "type" (costs, valuations, preferences), the game has incomplete information.

**Perfect information** means knowing what has happened so far: whenever a player moves,
they know every previous move by every player. If a player ever has to act without seeing
what someone else did (simultaneous moves, hidden actions, hidden cards), the game has
imperfect information.


|            |              Perfect              |            Imperfect            |
|------------|:---------------------------------:|:-------------------------------:|
|  Complete  | Tic-Tac-Toe, chess, Go, Connect 4 | Rock-Paper-Scissors, Battleship |
| Incomplete |       Haggling at a market        | Poker, Bridge, Catan, Starcraft |

## 2. Minimax

In [ ]:
import math


def solve(state):
    val = -math.inf
    best_action = None
    for next_state, action in state.succ_fn():
        result = min_value(next_state)
        if result > val:
            val = result
            best_action = action
    return best_action

def min_value(state):
    if state.is_terminal():
        return state.utility()

    val = math.inf
    for next_state, action in state.succ_fn():
        val = min(val, max_value(next_state))
    return val

def max_value(state):
    if state.is_terminal():
        return state.utility()

    val = -math.inf
    for next_state, action in state.succ_fn():
        val = max(val, min_value(next_state))
    return val

| Complete | Optimal |       Time       |  Space  |
|:--------:|:-------:|:----------------:|:-------:|
|   Yes*   |  Yes**  | O(b<sup>m</sup>) | O(b·m)  |

> \* Only if the game tree is finite.
>
> \*\* Only against an optimal opponent.


## 3. Alpha-Beta Pruning

In [ ]:
import math


def solve(state):
    val = -math.inf
    best_action = None
    for next_state, action in state.succ_fn():
        result = min_value(next_state, -math.inf, math.inf)
        if result > val:
            val = result
            best_action = action
    return best_action

def min_value(state, alpha, beta):
    if state.is_terminal():
        return state.utility()

    val = math.inf
    for next_state, action in state.succ_fn():
        val = min(val, max_value(next_state, alpha, beta))
        if val <= alpha:
            return val
        beta = min(beta, val)
    return val

def max_value(state, alpha , beta):
    if state.is_terminal():
        return state.utility()

    val = -math.inf
    for next_state, action in state.succ_fn():
        val = max(val, min_value(next_state, alpha, beta))
        if val >= beta:
            return val
        alpha = max(alpha, val)
    return val

- **Alpha** – the best value the Max player can already guarantee on the path to the current node.
- **Beta** – the best value the Min player can already guarantee on the path to the current node.

The search starts with the window (-**∞**, **∞**) and looks for values in the **open**
interval **(alpha, beta)**. The returned `val` tells us the following about the node's exact value:

- `val ≤ alpha` – the **exact value** ≤ `val`.
- `alpha < val < beta` – the **exact value** = `val`.
- `val ≥ beta` – the **exact value** ≥ `val`.

🔑 **Pruning does not change the final result.**

With perfect move ordering, the time complexity drops to **O(b<sup>m/2</sup>)**.

## 4. Cutoff and Evaluation

A real game tree is usually too large to search to the end, so we change Minimax in two ways:

- The **terminal test** becomes a **cutoff test**, such as a depth limit.
- The **utility function** becomes an **evaluation function**, which estimates how good a position is.

For chess, this is typically a weighted sum of features:
Eval(s) = w<sub>1</sub>f<sub>1</sub>(s) + ... + w<sub>n</sub>f<sub>n</sub>(s).

[Official Materials](../../official-materials/games.pdf#page=23)

## 5. Homework - Tic-Tac-Toe

### 🔑 5.1 Tic-Tac-Toe Is a Zero-Sum Game

Because the game is zero-sum, one player's score is the negative of the other's:

MinScore = -MaxScore

This lets us replace `min_value` and `max_value` with a single function, **negamax**, which
always maximizes from the current player's point of view. Switching players negates the
window, so (alpha, beta) becomes (-beta, -alpha).

In [ ]:
def solve(state):
    val = -math.inf
    best_action = None
    for next_state, action in state.succ_fn():
        result = -negamax(next_state, -math.inf, math.inf)
        if result > val:
            val = result
            best_action = action
    return best_action

def negamax(state, alpha:float | int , beta: float | int) -> float | int:
    if state.is_terminal():
        # The utility must be from the current player's point of view.
        return state.utility()

    val = -math.inf
    for next_state, action in state.succ_fn():
        val = max(val, -negamax(next_state, -beta, -alpha))
        if val >= beta:
            return val
        alpha = max(alpha, val)
    return val

### 5.2 Tic-Tac-Toe Optimality

Naive Tic-Tac-Toe scoring:

- `1` – player 1 wins
- `0` – draw
- `-1` – player 2 wins

With this scoring, the algorithm is **not optimal**: an immediate win and a forced win
later both score `1`, so it may skip the immediate win.
See the [naive scoring example](../../official-materials/games.pdf#page=31).

To fix this, we make the score depend on the depth, so that a faster win scores higher.
Here, an optimal algorithm is one that finds the shallowest win.

*depth* ∈ [0, 9] is the number of non-empty cells:

- `1` => `10 - depth`
- `0` => `0`
- `-1` => `depth - 10`

See the [optimal scoring example](../../official-materials/games.pdf#page=46).